# Direct PDE Parameter Optimization

Optimizes tumor PDE parameters (diffusivity, proliferation, treatment response)
directly against the target scan using the finite-volume solver. No neural
network — just scipy.optimize over 2-3 physics parameters.

**Setup:**
Data should be in Google Drive at `My Drive/STS_Project_Data/data/`

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the repo and install
!rm -rf /content/BINNs
!git clone https://github.com/tanushappapogu-max/BINNs.git /content/BINNs
%cd /content/BINNs
!pip install -e '.[ml,imaging]' -q

In [ ]:
# Verify setup
import numpy as np
from scipy.optimize import minimize
print(f'NumPy: {np.__version__}')
print('scipy.optimize ready')

In [ ]:
# Configure paths
from pathlib import Path

DRIVE_DATA = Path('/content/drive/MyDrive/STS_Project_Data/data')
NIFTI_ROOT = DRIVE_DATA / 'raw' / 'mu_glioma_post'
DERIVED = DRIVE_DATA / 'derived' / 'mu_glioma_post'
MANIFEST = DERIVED / 'shared_split_manifest.json'

ROLE = 'model_selection'  # 'model_selection' for validation (39), 'training' for full (208)
TRANSITION_INDEX = DERIVED / f'shared_{ROLE}_transitions.json'
OUTPUT_ROOT = DRIVE_DATA / f'outputs/direct_optim_{ROLE}'

for p in [NIFTI_ROOT, MANIFEST, TRANSITION_INDEX]:
    assert p.exists(), f'Missing: {p}'
print('All paths verified')

In [ ]:
import json
from gbm_pinn.pinn_cohort import PINNCohortConfig, run_pinn_cohort

config = PINNCohortConfig(
    transition_index_path=TRANSITION_INDEX,
    manifest_path=MANIFEST,
    nifti_root=NIFTI_ROOT,
    output_root=OUTPUT_ROOT,
    data_root=Path('/content/drive/MyDrive/STS_Project_Data'),
    role=ROLE,
    resume=True,
)

summary = run_pinn_cohort(config)
print(json.dumps(summary, indent=2))

In [ ]:
# Analyze results
import json
summary = json.load(open(OUTPUT_ROOT / 'cohort_summary.json'))
print(f"Mean Dice: {summary['mean_dice']:.4f}")
print(f"Mean Skill: {summary['mean_dice_skill_over_persistence']:+.4f}")
print(f"Beating Persistence: {summary['n_beating_persistence']}/{summary['n_successful']}")